# Hospedando agente LangGraph com modelos do Amazon Bedrock no Amazon Bedrock AgentCore Runtime

## Visão Geral

Neste tutorial, aprenderemos como hospedar seu agente existente usando o Amazon Bedrock AgentCore Runtime. 

Focaremos em um exemplo de LangGraph com modelo do Amazon Bedrock. Para Strands Agents com modelo do Amazon Bedrock, consulte [aqui](../01-strands-with-bedrock-model)
e para Strands Agents com um modelo OpenAI, consulte [aqui](../03-strands-with-openai-model).

### Detalhes do Tutorial

| Informação         | Detalhes                                                                      |
|:--------------------|:-----------------------------------------------------------------------------|
| Tipo de tutorial       | Conversacional                                                               |
| Tipo de agente          | Único                                                                       |
| Framework Agêntico   | LangGraph                                                                    |
| Modelo LLM           | Anthropic Claude Haiku 4.5                                                  |
| Componentes | Hospedagem de agente no AgentCore Runtime. Usando LangGraph e Modelo Amazon Bedrock |
| Vertical do tutorial   | Multisetorial                                                               |
| Complexidade  | Fácil                                                                         |
| SDK utilizado            | Amazon BedrockAgentCore Python SDK e boto3                                 |

### Arquitetura do Tutorial

Neste tutorial, descreveremos como implantar um agente existente no AgentCore runtime. 

Para fins de demonstração, usaremos um agente LangGraph usando modelos do Amazon Bedrock

Em nosso exemplo, usaremos um agente muito simples com duas ferramentas: `get_weather` e `get_time`. 

<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="50%"/>
</div>

### Principais Recursos do Tutorial

* Hospedando Agentes no Amazon Bedrock AgentCore Runtime
* Usando modelos do Amazon Bedrock
* Usando LangGraph


## Pré-requisitos

Para executar este tutorial, você precisará de:
* Python 3.10+
* Credenciais AWS
* Amazon Bedrock AgentCore SDK
* LangGraph
* Docker em execução

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Criando seus agentes e experimentando localmente

Antes de implantar nossos agentes no AgentCore Runtime, vamos desenvolvê-los e executá-los localmente para fins de experimentação.

Para aplicações agênticas de produção, precisaremos desacoplar o processo de criação do agente do processo de invocação. Com o AgentCore Runtime, decoraremos a parte de invocação do nosso agente com o decorador `@app.entrypoint` e o teremos como ponto de entrada para nosso runtime. Vamos primeiro ver como cada agente é desenvolvido durante a fase de experimentação.

A arquitetura aqui será a seguinte:

<div style="text-align:left">
    <img src="images/architecture_local.png" width="60%"/>
</div>

In [ ]:
%%writefile langgraph_bedrock.py
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
import argparse
import json
import operator
import math

# Create calculator tool
@tool
def calculator(expression: str) -> str:
    """
    Calculate the result of a mathematical expression.
    
    Args:
        expression: A mathematical expression as a string (e.g., "2 + 3 * 4", "sqrt(16)", "sin(pi/2)")
    
    Returns:
        The result of the calculation as a string
    """
    try:
        # Define safe functions that can be used in expressions
        safe_dict = {
            "__builtins__": {},
            "abs": abs, "round": round, "min": min, "max": max,
            "sum": sum, "pow": pow,
            # Math functions
            "sqrt": math.sqrt, "sin": math.sin, "cos": math.cos, "tan": math.tan,
            "log": math.log, "log10": math.log10, "exp": math.exp,
            "pi": math.pi, "e": math.e,
            "ceil": math.ceil, "floor": math.floor,
            "degrees": math.degrees, "radians": math.radians,
            # Basic operators (for explicit use)
            "add": operator.add, "sub": operator.sub,
            "mul": operator.mul, "truediv": operator.truediv,
        }
        
        # Evaluate the expression safely
        result = eval(expression, safe_dict)
        return str(result)
        
    except ZeroDivisionError:
        return "Error: Division by zero"
    except ValueError as e:
        return f"Error: Invalid value - {str(e)}"
    except SyntaxError:
        return "Error: Invalid mathematical expression"
    except Exception as e:
        return f"Error: {str(e)}"

# Create a custom weather tool
@tool
def weather():
    """Get weather"""  # Dummy implementation
    return "sunny"

# Define the agent using manual LangGraph construction
def create_agent():
    """Create and configure the LangGraph agent"""
    from langchain_aws import ChatBedrock
    
    # Initialize your LLM (adjust model and parameters as needed)
    llm = ChatBedrock(
        model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # or your preferred model
        model_kwargs={"temperature": 0.1}
    )
    
    # Bind tools to the LLM
    tools = [calculator, weather]
    llm_with_tools = llm.bind_tools(tools)
    
    # System message
    system_message = "You're a helpful assistant. You can do simple math calculation, and tell the weather."
    
    # Define the chatbot node
    def chatbot(state: MessagesState):
        # Add system message if not already present
        messages = state["messages"]
        if not messages or not isinstance(messages[0], SystemMessage):
            messages = [SystemMessage(content=system_message)] + messages
        
        response = llm_with_tools.invoke(messages)
        return {"messages": [response]}
    
    # Create the graph
    graph_builder = StateGraph(MessagesState)
    
    # Add nodes
    graph_builder.add_node("chatbot", chatbot)
    graph_builder.add_node("tools", ToolNode(tools))
    
    # Add edges
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )
    graph_builder.add_edge("tools", "chatbot")
    
    # Set entry point
    graph_builder.set_entry_point("chatbot")
    
    # Compile the graph
    return graph_builder.compile()

# Initialize the agent
agent = create_agent()

def langgraph_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    
    # Create the input in the format expected by LangGraph
    response = agent.invoke({"messages": [HumanMessage(content=user_input)]})
    
    # Extract the final message content
    return response["messages"][-1].content

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    response = langgraph_bedrock(json.loads(args.payload))
    print(response)

#### Invocando agente local

In [ ]:
!python langgraph_bedrock.py '{"prompt": "What is the weather now?"}'

## Preparando seu agente para implantação no AgentCore Runtime

Vamos agora implantar nossos agentes no AgentCore Runtime. Para isso, precisamos:
* Importar o Runtime App com `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Inicializar o App em nosso código com `app = BedrockAgentCoreApp()`
* Decorar a função de invocação com o decorador `@app.entrypoint`
* Deixar o AgentCoreRuntime controlar a execução do agente com `app.run()`

### LangGraph com modelo do Amazon Bedrock
Vamos começar com nosso LangGraph usando modelo do Amazon Bedrock. Outros exemplos com diferentes frameworks e modelos estão disponíveis nos diretórios pai

In [ ]:
%%writefile langgraph_bedrock.py
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from bedrock_agentcore.runtime import BedrockAgentCoreApp
import argparse
import json
import operator
import math

app = BedrockAgentCoreApp()

# Create calculator tool
@tool
def calculator(expression: str) -> str:
    """
    Calculate the result of a mathematical expression.
    
    Args:
        expression: A mathematical expression as a string (e.g., "2 + 3 * 4", "sqrt(16)", "sin(pi/2)")
    
    Returns:
        The result of the calculation as a string
    """
    try:
        # Define safe functions that can be used in expressions
        safe_dict = {
            "__builtins__": {},
            "abs": abs, "round": round, "min": min, "max": max,
            "sum": sum, "pow": pow,
            # Math functions
            "sqrt": math.sqrt, "sin": math.sin, "cos": math.cos, "tan": math.tan,
            "log": math.log, "log10": math.log10, "exp": math.exp,
            "pi": math.pi, "e": math.e,
            "ceil": math.ceil, "floor": math.floor,
            "degrees": math.degrees, "radians": math.radians,
            # Basic operators (for explicit use)
            "add": operator.add, "sub": operator.sub,
            "mul": operator.mul, "truediv": operator.truediv,
        }
        
        # Evaluate the expression safely
        result = eval(expression, safe_dict)
        return str(result)
        
    except ZeroDivisionError:
        return "Error: Division by zero"
    except ValueError as e:
        return f"Error: Invalid value - {str(e)}"
    except SyntaxError:
        return "Error: Invalid mathematical expression"
    except Exception as e:
        return f"Error: {str(e)}"

# Create a custom weather tool
@tool
def weather():
    """Get weather"""  # Dummy implementation
    return "sunny"

# Define the agent using manual LangGraph construction
def create_agent():
    """Create and configure the LangGraph agent"""
    from langchain_aws import ChatBedrock
    
    # Initialize your LLM (adjust model and parameters as needed)
    llm = ChatBedrock(
        model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # or your preferred model
        model_kwargs={"temperature": 0.1}
    )
    
    # Bind tools to the LLM
    tools = [calculator, weather]
    llm_with_tools = llm.bind_tools(tools)
    
    # System message
    system_message = "You're a helpful assistant. You can do simple math calculation, and tell the weather."
    
    # Define the chatbot node
    def chatbot(state: MessagesState):
        # Add system message if not already present
        messages = state["messages"]
        if not messages or not isinstance(messages[0], SystemMessage):
            messages = [SystemMessage(content=system_message)] + messages
        
        response = llm_with_tools.invoke(messages)
        return {"messages": [response]}
    
    # Create the graph
    graph_builder = StateGraph(MessagesState)
    
    # Add nodes
    graph_builder.add_node("chatbot", chatbot)
    graph_builder.add_node("tools", ToolNode(tools))
    
    # Add edges
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )
    graph_builder.add_edge("tools", "chatbot")
    
    # Set entry point
    graph_builder.set_entry_point("chatbot")
    
    # Compile the graph
    return graph_builder.compile()

# Initialize the agent
agent = create_agent()

@app.entrypoint
def langgraph_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    
    # Create the input in the format expected by LangGraph
    response = agent.invoke({"messages": [HumanMessage(content=user_input)]})
    
    # Extract the final message content
    return response["messages"][-1].content

if __name__ == "__main__":
    app.run()

## O que acontece nos bastidores?

Quando você usa `BedrockAgentCoreApp`, ele automaticamente:

* Cria um servidor HTTP que escuta na porta 8080
* Implementa o endpoint `/invocations` necessário para processar os requisitos do agente
* Implementa o endpoint `/ping` para verificações de integridade (muito importante para agentes assíncronos)
* Gerencia tipos de conteúdo e formatos de resposta adequados
* Gerencia tratamento de erros de acordo com os padrões AWS

## Implantando o agente no AgentCore Runtime

A operação `CreateAgentRuntime` suporta opções abrangentes de configuração, permitindo especificar imagens de contêiner, variáveis de ambiente e configurações de criptografia. Você também pode configurar configurações de protocolo (HTTP, MCP) e mecanismos de autorização para controlar como seus clientes se comunicam com o agente. 

**Nota:** A melhor prática de operações é empacotar o código como contêiner e enviar para o ECR usando pipelines CI/CD e IaC

Neste tutorial, usaremos o Amazon Bedrock AgentCode Python SDK para empacotar facilmente seus artefatos e implantá-los no AgentCore runtime.

### Configurar implantação do AgentCore Runtime

Primeiro, usaremos nosso kit inicial para configurar a implantação do AgentCore Runtime com um ponto de entrada, a função de execução que acabamos de criar e um arquivo de requisitos. Também configuraremos o kit inicial para criar automaticamente o repositório Amazon ECR no lançamento.

Durante a etapa de configuração, seu arquivo docker será gerado com base no código do seu aplicativo

<div style="text-align:left">
    <img src="images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()

agent_name = "langgraph_claude_getting_started"
response = agentcore_runtime.configure(
    entrypoint="langgraph_bedrock.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name
)
response

### Lançando agente no AgentCore Runtime

Agora que temos um arquivo docker, vamos lançar o agente no AgentCore Runtime. Isso criará o repositório Amazon ECR e o AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [ ]:
launch_result = agentcore_runtime.launch()

### Verificando o Status do AgentCore Runtime
Agora que implantamos o AgentCore Runtime, vamos verificar seu status de implantação

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### Invocando AgentCore Runtime

Finalmente, podemos invocar nosso AgentCore Runtime com um payload

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "How much is 2+2?"})
invoke_response

### Processando resultados da invocação

Agora podemos processar nossos resultados de invocação para incluí-los em um aplicativo

In [ ]:
from IPython.display import Markdown, display
import json
response_text = invoke_response['response'][0]
display(Markdown(response_text))

### Invocando AgentCore Runtime com boto3

Agora que seu AgentCore Runtime foi criado, você pode invocá-lo com qualquer AWS SDK. Por exemplo, você pode usar o método `invoke_agent_runtime` do boto3 para isso.

In [ ]:
import boto3
agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What is 2+2?"})
)

# Capture the runtime session ID for lifecycle management
runtime_session_id = boto3_response.get('runtimeSessionId')
print(f"Runtime Session ID: {runtime_session_id}")

if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads(events[0].decode("utf-8"))))

### Parando uma Sessão 

Você vai querer parar sessões individuais quando elas não forem mais necessárias.
Isso libera os recursos de microVM para essa sessão enquanto mantém o runtime ativo
para novas sessões. Abaixo demonstramos `stop_runtime_session`.

In [ ]:
# --- Inline Session Lifecycle Demo ---
# stop_runtime_session releases the microVM resources for this specific session while keeping the runtime alive for new sessions.


if runtime_session_id:
    agentcore_client.stop_runtime_session(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=runtime_session_id,
        qualifier='DEFAULT'
    )
    print(f"✅ Session '{runtime_session_id}' stopped — microVM resources released")
else:
    print("⚠️ No session ID available to stop")

### Demonstração de Configuração de Ciclo de Vida 

Agora vamos demonstrar como configurar um runtime com um tempo limite de inatividade mais curto.
Criaremos um segundo runtime com um tempo limite de inatividade de 5 minutos (300 segundos) para mostrar
como a configuração de ciclo de vida afeta o comportamento da sessão. Ambos os runtimes coexistirão.

In [ ]:
# --- Lifecycle Configuration Demo ---
# In production, choose a timeout appropriate for your workload:
#   - Development/testing: 5-15 minutes
#   - Interactive sessions: 30-60 minutes
#   - Long-running workloads: adjust as needed
#

agentcore_runtime_short = Runtime()
agent_name_short = "langgraph_claude_short_timeout"

# Configure with shorter idle timeout
response_short = agentcore_runtime_short.configure(
    entrypoint="langgraph_bedrock.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name_short
)

# Launch the second runtime
launch_result_short = agentcore_runtime_short.launch()
print(f"Second runtime launched: {launch_result_short.agent_id}")

# Wait for it to be ready
status_response_short = agentcore_runtime_short.status()
status_short = status_response_short.endpoint['status']
while status_short not in ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']:
    time.sleep(10)
    status_response_short = agentcore_runtime_short.status()
    status_short = status_response_short.endpoint['status']
    print(f"Short timeout runtime status: {status_short}")

# Now update the runtime with shorter idle timeout using boto3
# UpdateAgentRuntime is a full-replacement API — we must re-supply all required fields.
# First, retrieve the current runtime configuration.
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
current_runtime = agentcore_control_client.get_agent_runtime(
    agentRuntimeId=launch_result_short.agent_id
)

update_response = agentcore_control_client.update_agent_runtime(
    agentRuntimeId=launch_result_short.agent_id,
    agentRuntimeArtifact=current_runtime['agentRuntimeArtifact'],
    roleArn=current_runtime['roleArn'],
    networkConfiguration=current_runtime['networkConfiguration'],
    lifecycleConfiguration={
        'idleRuntimeSessionTimeout': 300  # 5 minutes
    }
)
print(f"✅ Runtime updated with 5-minute idle timeout")

# Invoke the second runtime to verify it works
invoke_response_short = agentcore_runtime_short.invoke({"prompt": "What is 3+3?"})
print(f"Second runtime response: {invoke_response_short['response'][0]}")

## Limpeza

Vamos agora limpar o AgentCore Runtime e recursos associados. Deletamos o runtime primeiro para evitar custos indesejados, depois limpamos recursos de suporte como repositórios ECR.

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

In [ ]:
# --- Stop active sessions to release microVM resources ---
import boto3

agentcore_client = boto3.client('bedrock-agentcore', region_name=region)
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
ecr_client = boto3.client('ecr', region_name=region)

# Stop the active session to release its microVM resources
# In production, this is how you end individual user sessions while keeping the runtime alive
# AgentCore Runtime costs are based on vCPU and Memory — stopping sessions avoids undesired costs
# Note: If the session was already stopped in the earlier demo cell, this will raise a
# ResourceNotFoundException — the except block handles that gracefully.
if 'runtime_session_id' in locals() and runtime_session_id:
    try:
        agentcore_client.stop_runtime_session(
            agentRuntimeArn=launch_result.agent_arn,
            runtimeSessionId=runtime_session_id,
            qualifier='DEFAULT'
        )
        print(f"✅ Session '{runtime_session_id}' stopped")
    except Exception as e:
        print(f"⚠️ Failed to stop session '{runtime_session_id}': {e}")

# --- Delete both runtimes ---
# Original runtime
try:
    agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result.agent_id,
    )
    print(f"✅ Original runtime '{launch_result.agent_id}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete original runtime: {e}")

# Short-timeout runtime
if 'launch_result_short' in locals():
    try:
        agentcore_control_client.delete_agent_runtime(
            agentRuntimeId=launch_result_short.agent_id,
        )
        print(f"✅ Short-timeout runtime '{launch_result_short.agent_id}' deleted")
    except Exception as e:
        print(f"⚠️ Failed to delete short-timeout runtime: {e}")

# --- Delete ECR repositories ---
try:
    ecr_client.delete_repository(
        repositoryName=launch_result.ecr_uri.split('/')[1],
        force=True
    )
    print(f"✅ ECR repository '{launch_result.ecr_uri.split('/')[1]}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete ECR repository: {e}")

if 'launch_result_short' in locals():
    try:
        ecr_client.delete_repository(
            repositoryName=launch_result_short.ecr_uri.split('/')[1],
            force=True
        )
        print(f"✅ Second ECR repository '{launch_result_short.ecr_uri.split('/')[1]}' deleted")
    except Exception as e:
        print(f"⚠️ Failed to delete second ECR repository: {e}")

# Parabéns!